In [ ]:
#Basic LORA SETUP and Pipeline Creation

In [ ]:

import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1e9, 2
    ))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU detected. Check Colab runtime settings.")


Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 15.64
CUDA version: 13.0


In [ ]:

%pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 datasets==3.1.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 105.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requ

In [ ]:

import torch
import transformers
import peft
import accelerate
import datasets
import huggingface_hub

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not detected")
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Transformers: 4.46.3
PEFT: 0.13.2
Accelerate: 1.1.1
Datasets: 3.1.0
Hugging Face Hub: 0.36.2


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)
model = model.to("cuda")
model.eval()

print("Model loaded:", MODEL_NAME)
print("Model device:", next(model.parameters()).device)
print("Tokenizer vocabulary size:", len(tokenizer))
print("Model parameters (millions):",
      round(sum(p.numel() for p in model.parameters()) / 1e6, 1))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B
Model device: cuda:0
Tokenizer vocabulary size: 151665
Model parameters (millions): 494.0


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# LoRA configuration for our first smoke test
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# Display trainable vs. total parameters
model.print_trainable_parameters()


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [ ]:
trainable = []
frozen_count = 0

for name, param in model.named_parameters():
    if param.requires_grad:
        trainable.append((name, param.numel()))
    else:
        frozen_count += param.numel()

print("Trainable tensors:", len(trainable))
print("Trainable parameters:", sum(n for _, n in trainable))
print("Frozen parameters:", frozen_count)

print("\nFirst trainable parameter names:")
for name, _ in trainable[:12]:
    print(name)


Trainable tensors: 192
Trainable parameters: 1081344
Frozen parameters: 494032768

First trainable parameter names:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.k_proj.lora_B.default.weight


In [ ]:
examples = [
    {"prompt": "What is 2 + 3?", "answer": "2 + 3 = 5."},
    {"prompt": "What is 7 + 5?", "answer": "7 + 5 = 12."},
    {"prompt": "What is 9 - 4?", "answer": "9 - 4 = 5."},
    {"prompt": "What is 3 * 4?", "answer": "3 * 4 = 12."},
]

def format_example(example):
    return (
        f"Question: {example['prompt']}\n"
        f"Answer: {example['answer']}"
    )

texts = [format_example(ex) for ex in examples]
print(texts[0])


Question: What is 2 + 3?
Answer: 2 + 3 = 5.


In [ ]:
from torch.optim import AdamW

model.train()
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=2e-4,
)

def make_batch(example):
    prefix = f"Question: {example['prompt']}\nAnswer:"
    answer = " " + example["answer"]

    prefix_ids = tokenizer(
        prefix, add_special_tokens=True
    )["input_ids"]
    answer_ids = tokenizer(
        answer, add_special_tokens=False
    )["input_ids"]

    input_ids = prefix_ids + answer_ids + [tokenizer.eos_token_id]
    labels = [-100] * len(prefix_ids) + answer_ids + [tokenizer.eos_token_id]

    return (
        torch.tensor([input_ids], device="cuda"),
        torch.tensor([labels], device="cuda"),
    )

input_ids, labels = make_batch(examples[0])

optimizer.zero_grad(set_to_none=True)
outputs = model(input_ids=input_ids, labels=labels)
loss = outputs.loss
loss.backward()

# Check that LoRA received gradients
grad_norms = [
    p.grad.norm().item()
    for p in model.parameters()
    if p.requires_grad and p.grad is not None
]

print("Loss:", round(loss.item(), 4))
print("LoRA tensors with gradients:", len(grad_norms))
print("Gradient norm (sum):", round(sum(grad_norms), 4))

optimizer.step()
optimizer.zero_grad(set_to_none=True)

print("Optimizer step completed successfully.")


Loss: 0.5067
LoRA tensors with gradients: 192
Gradient norm (sum): 47.5789
Optimizer step completed successfully.


In [ ]:
import os

SAVE_DIR = "/content/qwen_lora_smoke_test"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved adapter to:", SAVE_DIR)
print("Files:", os.listdir(SAVE_DIR))


Saved adapter to: /content/qwen_lora_smoke_test
Files: ['README.md', 'tokenizer_config.json', 'added_tokens.json', 'special_tokens_map.json', 'merges.txt', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'vocab.json']
